# Train Canonical Evaluator Adapter: `SANCTIONS_SCREENING_MATCH`
**Telegraph Protocol Evaluator Adapter for ModernBERT-base Foundation Model**
- Intent: `SANCTIONS_SCREENING_MATCH`
- Class: `judgment`
- Foundation: `modernbert-base-lora16-v1`
- LoRA Config: rank 8, alpha 32, target modules `Wqkv, Wo, Wi`

In [ ]:
# 1. Setup Environment & Clone Baseline
!pip install -q transformers>=4.48 peft datasets onnx onnxruntime
!git clone https://github.com/telegraphprotocol/telegraph-wasm-baseline.git baseline
%cd baseline
!mkdir -p foundation
!curl -sSL https://github.com/telegraphprotocol/telegraph-foundation/releases/download/modernbert-base-lora16-v1/foundation.json -o foundation/foundation.json
!curl -sSL https://github.com/telegraphprotocol/telegraph-foundation/releases/download/modernbert-base-lora16-v1/tokenizer.json -o foundation/tokenizer.json
!curl -sSL https://github.com/telegraphprotocol/telegraph-foundation/releases/download/modernbert-base-lora16-v1/foundation_engine.wasm -o foundation/foundation_engine.wasm
!sha256sum foundation/*

In [ ]:
# 2. Download / Load Sanctions Preference Dataset
!curl -sSL https://raw.githubusercontent.com/VishAlgoDev/praman-desk/main/eval-data/pairs_sanctions.jsonl -o pairs_sanctions.jsonl
!head -n 2 pairs_sanctions.jsonl
!wc -l pairs_sanctions.jsonl

In [ ]:
# 3. Run Anti-Shortcut Dataset Gate Check
!python -m float_runtime.train_adapter --intent SANCTIONS_SCREENING_MATCH --data pairs_sanctions.jsonl --out runs/SANCTIONS_SCREENING_MATCH --check-data

In [ ]:
# 4. Train LoRA Evaluator Adapter on GPU
import torch
print('GPU Available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

!python -m float_runtime.train_adapter \
  --intent SANCTIONS_SCREENING_MATCH \
  --data pairs_sanctions.jsonl \
  --out runs/SANCTIONS_SCREENING_MATCH \
  --lora-rank 8 \
  --lora-alpha 32 \
  --epochs 4 \
  --batch-size 8 \
  --lr 5e-4

In [ ]:
# 5. Export Canonical LoRA safetensors Package
!python -m float_runtime.export_adapter \
  --run runs/SANCTIONS_SCREENING_MATCH \
  --intent SANCTIONS_SCREENING_MATCH \
  --class judgment \
  --foundation foundation/foundation.json

!ls -lh runs/SANCTIONS_SCREENING_MATCH

In [ ]:
# 6. Verify Keccak-256 Hash & Download safetensors
from google.colab import files
import json

with open('runs/SANCTIONS_SCREENING_MATCH/result.json') as f:
  res = json.load(f)
print(json.dumps(res, indent=2))

files.download('runs/SANCTIONS_SCREENING_MATCH/SANCTIONS_SCREENING_MATCH.safetensors')
files.download('runs/SANCTIONS_SCREENING_MATCH/result.json')